In [0]:
from delta.tables import *
from pyspark.sql import functions as F
# Compara o orçamento planejado por categoria vs. o gasto real

bronze_budgets    = spark.table("bronze.budgets")
bronze_expenses   = spark.table("bronze.expenses")
bronze_budget_cat = spark.table("bronze.budget_categories")

actual_per_budget = (
    bronze_expenses
    .groupBy("budget_id")
    .agg(F.sum("amount").alias("actual_cents"))
)

fact_budget = (
    bronze_budgets.alias("b")
    .join(bronze_budget_cat.alias("bc"),
          F.col("b.category_id") == F.col("bc.id"))
    .join(actual_per_budget.alias("a"),
          F.col("b.id") == F.col("a.budget_id"), "left")
    .select(
        F.col("b.id").alias("budget_id"),
        F.col("b.event_id"),
        F.col("bc.name").alias("category_name"),
        F.col("bc.parent_id"),
        F.col("b.max_amount").alias("budgeted_cents"),
        F.coalesce(F.col("a.actual_cents"), F.lit(0)).alias("actual_cents"),
        (F.col("b.max_amount") - F.coalesce(F.col("a.actual_cents"), F.lit(0)))
            .alias("remaining_cents"),
        (F.coalesce(F.col("a.actual_cents"), F.lit(0)) / F.col("b.max_amount"))
            .alias("utilization_rate"),
        F.col("b.status").alias("budget_status"),
    )
)

(fact_budget.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_bi.fact_budget_vs_actual"))